# 05_realistic_model

This notebook builds and solves the realistic Part 2 model in Gurobi.

**Main tasks**
1. Load realistic-model inputs.
2. Build the MILP with realistic expansion and siting rules.
3. Solve the model.
4. Export facility-level, new-build, and zipcode-level solution tables.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

In [3]:
def find_project_root():
    """Return a reasonable project root based on the current working directory."""
    cwd = Path.cwd()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for p in candidates:
        if (p / "data").exists():
            return p
    return cwd

PROJECT_ROOT = find_project_root()
IDEAL_DIR = PROJECT_ROOT / "data" / "processed" / "ideal"
REALISTIC_DIR = PROJECT_ROOT / "data" / "processed" / "realistic"

RESULTS_DIR = PROJECT_ROOT / "results" / "realistic"
LOG_DIR = RESULTS_DIR / "logs"
SOLUTIONS_DIR = RESULTS_DIR / "solutions"

LOG_DIR.mkdir(parents=True, exist_ok=True)
SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IDEAL_DIR:", IDEAL_DIR)
print("REALISTIC_DIR:", REALISTIC_DIR)
print("SOLUTIONS_DIR:", SOLUTIONS_DIR)

PROJECT_ROOT: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
IDEAL_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/ideal
REALISTIC_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/realistic
SOLUTIONS_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/solutions


In [4]:
required_files = [
    REALISTIC_DIR / "zipcode_demand_supply_realistic.csv",
    REALISTIC_DIR / "facility_expansion_params_realistic.csv",
    REALISTIC_DIR / "candidate_build_options_realistic.csv",
    REALISTIC_DIR / "candidate_candidate_conflicts.csv",
    REALISTIC_DIR / "candidate_existing_conflicts.csv",
]

missing_files = [str(f) for f in required_files if not f.exists()]

print("Required input files check:")
for f in required_files:
    print(f"  {f.name}: {f.exists()}")

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing. Run the realistic parameter notebook first:\n"
        + "\n".join(missing_files)
    )


Required input files check:
  zipcode_demand_supply_realistic.csv: True
  facility_expansion_params_realistic.csv: True
  candidate_build_options_realistic.csv: True
  candidate_candidate_conflicts.csv: True
  candidate_existing_conflicts.csv: True


In [5]:
zip_df = pd.read_csv(REALISTIC_DIR / "zipcode_demand_supply_realistic.csv")
fac_df = pd.read_csv(REALISTIC_DIR / "facility_expansion_params_realistic.csv")
cand_df = pd.read_csv(REALISTIC_DIR / "candidate_build_options_realistic.csv")
cc_conflict_df = pd.read_csv(REALISTIC_DIR / "candidate_candidate_conflicts.csv")
ce_conflict_df = pd.read_csv(REALISTIC_DIR / "candidate_existing_conflicts.csv")


In [6]:
zip_df["zipcode"] = zip_df["zipcode"].astype(str).str.zfill(5)
fac_df["zipcode"] = fac_df["zipcode"].astype(str).str.zfill(5)
cand_df["zipcode"] = cand_df["zipcode"].astype(str).str.zfill(5)

fac_df["facility_id"] = fac_df["facility_id"].astype(str)
cand_df["candidate_id"] = pd.to_numeric(cand_df["candidate_id"], errors="coerce").astype(int)
cand_df["size"] = cand_df["size"].astype(str)

if not cc_conflict_df.empty:
    cc_conflict_df["zipcode"] = cc_conflict_df["zipcode"].astype(str).str.zfill(5)
    cc_conflict_df["candidate_id_1"] = pd.to_numeric(cc_conflict_df["candidate_id_1"], errors="coerce").astype(int)
    cc_conflict_df["candidate_id_2"] = pd.to_numeric(cc_conflict_df["candidate_id_2"], errors="coerce").astype(int)

if not ce_conflict_df.empty:
    ce_conflict_df["zipcode"] = ce_conflict_df["zipcode"].astype(str).str.zfill(5)
    ce_conflict_df["candidate_id"] = pd.to_numeric(ce_conflict_df["candidate_id"], errors="coerce").astype(int)
    ce_conflict_df["facility_id"] = ce_conflict_df["facility_id"].astype(str)

numeric_cols_zip = ["req_total", "req_under5", "existing_total", "existing_under5", "gap_total", "gap_under5"]
for c in numeric_cols_zip:
    if c in zip_df.columns:
        zip_df[c] = pd.to_numeric(zip_df[c], errors="coerce").fillna(0).astype(int)

numeric_cols_fac = [
    "n_f",
    "b_f",
    "cap1",
    "cap2",
    "cap3",
    "U_f_realistic",
    "regime1_lb",
    "regime1_ub",
    "regime2_lb",
    "regime2_ub",
    "regime3_lb",
    "regime3_ub",
    "coef1",
    "coef2",
    "coef3",
]
for c in numeric_cols_fac:
    if c in fac_df.columns:
        fac_df[c] = pd.to_numeric(fac_df[c], errors="coerce").fillna(0)

numeric_cols_cand = ["new_total_capacity", "new_under5_capacity_max", "fixed_build_cost"]
for c in numeric_cols_cand:
    if c in cand_df.columns:
        cand_df[c] = pd.to_numeric(cand_df[c], errors="coerce").fillna(0)

print("zip_df:", zip_df.shape)
display(zip_df.head())

print("fac_df:", fac_df.shape)
display(fac_df.head())

print("cand_df:", cand_df.shape)
display(cand_df.head())

print("Candidate-candidate conflicts:", cc_conflict_df.shape)
print("Candidate-existing conflicts:", ce_conflict_df.shape)


zip_df: (311, 26)


,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before,req_total_original,req_under5_original,zero_child_population_adjustment
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609,24,9.0,58,472,1,1,1,667,496,0
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724,198,56.0,0,1230,0,1,1,3394,1428,0
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995,0,7.0,0,960,0,1,1,984,960,0
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263,0,2.0,0,289,0,1,1,232,289,0
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39,0,1.0,363,323,1,1,1,402,323,0


fac_df: (7712, 22)


,facility_id,facility_name,program_type,zipcode,latitude,longitude,missing_geo_flag,n_f,b_f,cap1,cap2,cap3,U_f_realistic,regime1_lb,regime1_ub,regime2_lb,regime2_ub,regime3_lb,regime3_ub,coef1,coef2,coef3
0,51349,"Valerio, Gladys",FDC,10474,40.818399,-73.888816,0,6,6,0,0,1,1,0,0,0,0,1,1,3533.333333,3733.333333,4333.333333
1,73544,YMCA OF GREATER NY,SACC,10017,40.753216,-73.970794,0,60,0,6,3,3,12,1,6,7,9,10,12,533.333333,733.333333,1333.333333
2,108407,"Almond Tree Group Family Day Care, LLC",GFDC,11225,NaN,NaN,1,16,12,1,1,1,3,1,1,2,2,3,3,1450.000000,1650.000000,2250.000000
3,111953,"Williams, Petal",GFDC,11226,NaN,NaN,1,16,12,1,1,1,3,1,1,2,2,3,3,1450.000000,1650.000000,2250.000000
4,126425,"Hernandez, Lidia",GFDC,10465,40.841228,-73.823572,0,12,12,1,0,1,2,1,1,0,1,2,2,1866.666667,2066.666667,2666.666667


cand_df: (85566, 10)


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost,blocked_by_existing
0,1,10001,40.741893,-74.000140,False,small,100,50,65000,0
1,1,10001,40.741893,-74.000140,False,medium,200,100,95000,0
2,1,10001,40.741893,-74.000140,False,large,400,200,115000,0
3,2,10001,40.752007,-74.005436,False,small,100,50,65000,0
4,2,10001,40.752007,-74.005436,False,medium,200,100,95000,0


Candidate-candidate conflicts: (11455, 4)
Candidate-existing conflicts: (4730, 4)


In [7]:
required_zip_cols = ["zipcode", "req_total", "req_under5", "existing_total", "existing_under5"]
required_fac_cols = [
    "facility_id",
    "zipcode",
    "n_f",
    "b_f",
    "regime1_lb",
    "regime1_ub",
    "regime2_lb",
    "regime2_ub",
    "regime3_lb",
    "regime3_ub",
    "coef1",
    "coef2",
    "coef3",
]
required_cand_cols = ["candidate_id", "zipcode", "size", "new_total_capacity", "new_under5_capacity_max", "fixed_build_cost"]

missing_zip_cols = [c for c in required_zip_cols if c not in zip_df.columns]
missing_fac_cols = [c for c in required_fac_cols if c not in fac_df.columns]
missing_cand_cols = [c for c in required_cand_cols if c not in cand_df.columns]

if missing_zip_cols:
    raise KeyError(f"Missing columns in zipcode_demand_supply_realistic.csv: {missing_zip_cols}")
if missing_fac_cols:
    raise KeyError(f"Missing columns in facility_expansion_params_realistic.csv: {missing_fac_cols}")
if missing_cand_cols:
    raise KeyError(f"Missing columns in candidate_build_options_realistic.csv: {missing_cand_cols}")


In [8]:
Z = zip_df["zipcode"].tolist()
F = fac_df["facility_id"].tolist()
candidate_sites = sorted(cand_df["candidate_id"].unique().tolist())
S = sorted(cand_df["size"].unique().tolist())

req_total = dict(zip(zip_df["zipcode"], zip_df["req_total"]))
req_under5 = dict(zip(zip_df["zipcode"], zip_df["req_under5"]))
existing_total = dict(zip(zip_df["zipcode"], zip_df["existing_total"]))
existing_under5 = dict(zip(zip_df["zipcode"], zip_df["existing_under5"]))

fac_zip = dict(zip(fac_df["facility_id"], fac_df["zipcode"]))
n_f = dict(zip(fac_df["facility_id"], fac_df["n_f"]))
b_f = dict(zip(fac_df["facility_id"], fac_df["b_f"]))
regime1_lb = dict(zip(fac_df["facility_id"], fac_df["regime1_lb"].astype(int)))
regime1_ub = dict(zip(fac_df["facility_id"], fac_df["regime1_ub"].astype(int)))
regime2_lb = dict(zip(fac_df["facility_id"], fac_df["regime2_lb"].astype(int)))
regime2_ub = dict(zip(fac_df["facility_id"], fac_df["regime2_ub"].astype(int)))
regime3_lb = dict(zip(fac_df["facility_id"], fac_df["regime3_lb"].astype(int)))
regime3_ub = dict(zip(fac_df["facility_id"], fac_df["regime3_ub"].astype(int)))
coef1 = dict(zip(fac_df["facility_id"], fac_df["coef1"]))
coef2 = dict(zip(fac_df["facility_id"], fac_df["coef2"]))
coef3 = dict(zip(fac_df["facility_id"], fac_df["coef3"]))

cand_key_tuples = list(zip(cand_df["candidate_id"], cand_df["size"]))

build_total = {(r["candidate_id"], r["size"]): r["new_total_capacity"] for _, r in cand_df.iterrows()}
build_under5_max = {(r["candidate_id"], r["size"]): r["new_under5_capacity_max"] for _, r in cand_df.iterrows()}
build_cost = {(r["candidate_id"], r["size"]): r["fixed_build_cost"] for _, r in cand_df.iterrows()}
cand_zip = {(r["candidate_id"], r["size"]): r["zipcode"] for _, r in cand_df.iterrows()}

fac_by_zip = fac_df.groupby("zipcode")["facility_id"].apply(list).to_dict()
candsize_by_zip = cand_df.groupby("zipcode").apply(
    lambda g: list(zip(g["candidate_id"], g["size"]))
).to_dict()
sizes_by_candidate = cand_df.groupby("candidate_id")["size"].apply(list).to_dict()

blocked_candidates = set(ce_conflict_df["candidate_id"].unique().tolist()) if not ce_conflict_df.empty else set()

print("Number of zipcodes:", len(Z))
print("Number of facilities:", len(F))
print("Number of candidate sites:", len(candidate_sites))
print("Number of candidate-size options:", len(cand_key_tuples))
print("Blocked candidates from diagnostics:", len(blocked_candidates))


/var/folders/qb/g17v2g6s5hb2vnxkq2tq47_r0000gn/T/ipykernel_30712/4029030829.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  candsize_by_zip = cand_df.groupby("zipcode").apply(


Number of zipcodes: 311
Number of facilities: 7712
Number of candidate sites: 28522
Number of candidate-size options: 85566
Blocked candidates from diagnostics: 2578


## Decision variables

- `x1[f]`, `x2[f]`, `x3[f]`: total expansion chosen for facility `f` if it falls in the
  `0-10%`, `10-15%`, or `15-20%` cost regime
- `z1[f]`, `z2[f]`, `z3[f]`: binaries selecting at most one realistic expansion cost regime
- `u[f]`: under-5 slots assigned within expansion at facility `f`
- `y[l,s]`: binary variable equal to 1 if candidate site `l` is used for size `s`
- `g[l,s]`: under-5 slots assigned to the new facility built at candidate site `l` with size `s`


In [9]:
log_file = LOG_DIR / "realistic_model.log"

model = gp.Model("realistic_childcare")
model.Params.LogFile = str(log_file)
model.Params.MIPGap = 0.001
model.Params.TimeLimit = 600
model.Params.OutputFlag = 1

x1 = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x1")
x2 = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x2")
x3 = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x3")
z1 = model.addVars(F, vtype=GRB.BINARY, name="z1")
z2 = model.addVars(F, vtype=GRB.BINARY, name="z2")
z3 = model.addVars(F, vtype=GRB.BINARY, name="z3")
u = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="u")

y = model.addVars(cand_key_tuples, vtype=GRB.BINARY, name="y")
g = model.addVars(cand_key_tuples, vtype=GRB.INTEGER, lb=0, name="g")


Set parameter Username


Set parameter LicenseID to value 2773994


Academic license - for non-commercial use only - expires 2027-02-02


Set parameter LogFile to value "/Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/logs/realistic_model.log"


Set parameter MIPGap to value 0.001


Set parameter TimeLimit to value 600


Set parameter OutputFlag to value 1


In [10]:
for f in F:
    model.addConstr(z1[f] + z2[f] + z3[f] <= 1, name=f"one_regime_{f}")

    model.addConstr(x1[f] <= regime1_ub[f] * z1[f], name=f"regime1_ub_{f}")
    model.addConstr(x2[f] <= regime2_ub[f] * z2[f], name=f"regime2_ub_{f}")
    model.addConstr(x3[f] <= regime3_ub[f] * z3[f], name=f"regime3_ub_{f}")

    if regime1_ub[f] > 0:
        model.addConstr(x1[f] >= regime1_lb[f] * z1[f], name=f"regime1_lb_{f}")
    else:
        model.addConstr(z1[f] == 0, name=f"regime1_disabled_{f}")

    if regime2_ub[f] >= regime2_lb[f] and regime2_lb[f] > 0:
        model.addConstr(x2[f] >= regime2_lb[f] * z2[f], name=f"regime2_lb_{f}")
    else:
        model.addConstr(z2[f] == 0, name=f"regime2_disabled_{f}")

    if regime3_ub[f] >= regime3_lb[f] and regime3_lb[f] > 0:
        model.addConstr(x3[f] >= regime3_lb[f] * z3[f], name=f"regime3_lb_{f}")
    else:
        model.addConstr(z3[f] == 0, name=f"regime3_disabled_{f}")

    model.addConstr(u[f] <= x1[f] + x2[f] + x3[f], name=f"under5_expand_cap_{f}")


In [11]:
for l in candidate_sites:
    candidate_sizes = sizes_by_candidate.get(l, [])
    if candidate_sizes:
        model.addConstr(
            gp.quicksum(y[l, s] for s in candidate_sizes) <= 1,
            name=f"one_size_per_candidate_{l}"
        )

In [12]:
for (l, s) in cand_key_tuples:
    model.addConstr(
        g[l, s] <= build_under5_max[(l, s)] * y[l, s],
        name=f"new_under5_cap_{l}_{s}"
    )

In [13]:
for l in blocked_candidates:
    if l in sizes_by_candidate:
        model.addConstr(
            gp.quicksum(y[l, s] for s in sizes_by_candidate[l]) == 0,
            name=f"blocked_candidate_{l}"
        )

In [14]:
if not cc_conflict_df.empty:
    for _, row in cc_conflict_df.iterrows():
        l1 = int(row["candidate_id_1"])
        l2 = int(row["candidate_id_2"])

        sizes1 = sizes_by_candidate.get(l1, [])
        sizes2 = sizes_by_candidate.get(l2, [])

        if sizes1 and sizes2:
            model.addConstr(
                gp.quicksum(y[l1, s] for s in sizes1) + gp.quicksum(y[l2, s] for s in sizes2) <= 1,
                name=f"cc_conflict_{l1}_{l2}"
            )

In [15]:
for z in Z:
    expand_total_expr = gp.quicksum(
        x1[f] + x2[f] + x3[f]
        for f in fac_by_zip.get(z, [])
    )

    new_total_expr = gp.quicksum(
        build_total[(l, s)] * y[l, s]
        for (l, s) in candsize_by_zip.get(z, [])
    )

    model.addConstr(
        existing_total[z] + expand_total_expr + new_total_expr >= req_total[z],
        name=f"total_req_{z}"
    )

In [16]:
for z in Z:
    expand_under5_expr = gp.quicksum(
        u[f]
        for f in fac_by_zip.get(z, [])
    )

    new_under5_expr = gp.quicksum(
        g[l, s]
        for (l, s) in candsize_by_zip.get(z, [])
    )

    model.addConstr(
        existing_under5[z] + expand_under5_expr + new_under5_expr >= req_under5[z],
        name=f"under5_req_{z}"
    )

In [17]:
SPECIAL_EQUIPMENT_COST = 100

expansion_cost_expr = gp.quicksum(
    coef1[f] * x1[f] + coef2[f] * x2[f] + coef3[f] * x3[f]
    for f in F
)

new_build_cost_expr = gp.quicksum(
    build_cost[(l, s)] * y[l, s]
    for (l, s) in cand_key_tuples
)

under5_equipment_cost_expr = (
    gp.quicksum(SPECIAL_EQUIPMENT_COST * u[f] for f in F)
    + gp.quicksum(SPECIAL_EQUIPMENT_COST * g[l, s] for (l, s) in cand_key_tuples)
)

model.setObjective(
    expansion_cost_expr + new_build_cost_expr + under5_equipment_cost_expr,
    GRB.MINIMIZE
)

In [18]:
model.update()
print("Number of variables:", model.NumVars)
print("Number of constraints:", model.NumConstrs)

start_time = time.time()
model.optimize()
solve_seconds = time.time() - start_time

print("Status:", model.Status)
print("Solve time (sec):", round(solve_seconds, 2))
if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
    print("Objective value:", model.ObjVal)

Number of variables: 225116
Number of constraints: 186667
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D2128)


CPU model: Apple M4 Pro


Thread count: 12 physical cores, 12 logical processors, using up to 12 threads


Non-default parameters:


TimeLimit  600


...

Thread count was 12 (of 12 available processors)


Solution count 10: 1.85276e+08 1.85276e+08 1.85276e+08 ... 1.86905e+08


Time limit reached


Best objective 1.852759625313e+08, best bound 1.850124587928e+08, gap 0.1422%


Status: 9
Solve time (sec): 600.62
Objective value: 185275962.5313458


In [19]:
if model.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    model.computeIIS()
    iis_path = SOLUTIONS_DIR / "realistic_model.ilp"
    model.write(str(iis_path))
    print("IIS written to:", iis_path)

In [20]:
has_solution = model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] and model.SolCount > 0

if has_solution:
    facility_solution = fac_df.copy()
    facility_solution["x1"] = facility_solution["facility_id"].map(lambda f: x1[f].X)
    facility_solution["x2"] = facility_solution["facility_id"].map(lambda f: x2[f].X)
    facility_solution["x3"] = facility_solution["facility_id"].map(lambda f: x3[f].X)
    facility_solution["z1"] = facility_solution["facility_id"].map(lambda f: z1[f].X)
    facility_solution["z2"] = facility_solution["facility_id"].map(lambda f: z2[f].X)
    facility_solution["z3"] = facility_solution["facility_id"].map(lambda f: z3[f].X)
    facility_solution["u_under5"] = facility_solution["facility_id"].map(lambda f: u[f].X)
    facility_solution["expand_total"] = facility_solution["x1"] + facility_solution["x2"] + facility_solution["x3"]
    facility_solution["selected_regime"] = np.select(
        [
            facility_solution["z1"] > 0.5,
            facility_solution["z2"] > 0.5,
            facility_solution["z3"] > 0.5,
        ],
        [
            "0_10_pct",
            "10_15_pct",
            "15_20_pct",
        ],
        default="none",
    )
    facility_solution["expansion_cost"] = (
        facility_solution["coef1"] * facility_solution["x1"]
        + facility_solution["coef2"] * facility_solution["x2"]
        + facility_solution["coef3"] * facility_solution["x3"]
    )
    facility_solution["under5_equipment_cost_expansion"] = 100 * facility_solution["u_under5"]
    facility_solution["total_expansion_related_cost"] = (
        facility_solution["expansion_cost"] + facility_solution["under5_equipment_cost_expansion"]
    )
    facility_solution["expansion_share_of_existing"] = np.where(
        facility_solution["n_f"] > 0,
        facility_solution["expand_total"] / facility_solution["n_f"],
        0,
    )
    facility_solution = facility_solution[facility_solution["expand_total"] > 1e-6].copy()

    print("Expanded facilities:", facility_solution.shape)
    display(facility_solution.head())


Expanded facilities: (982, 35)


,facility_id,facility_name,program_type,zipcode,latitude,longitude,missing_geo_flag,n_f,b_f,cap1,cap2,cap3,U_f_realistic,regime1_lb,regime1_ub,regime2_lb,regime2_ub,regime3_lb,regime3_ub,coef1,coef2,coef3,x1,x2,x3,z1,z2,z3,u_under5,expand_total,selected_regime,expansion_cost,under5_equipment_cost_expansion,total_expansion_related_cost,expansion_share_of_existing
6,250057,KIPS BAY BOYS & GIRLS CLUB,SACC,10473,40.819564,-73.848827,0,240,0,24,12,12,48,1,24,25,36,37,48,283.333333,483.333333,1083.333333,24.0,0.0,0.0,1.0,0.0,0.0,24.0,24.0,0_10_pct,6800.000000,2400.0,9200.000000,0.100000
19,624233,"Directions For Our Youth, Inc.",SACC,10462,40.844890,-73.858149,0,245,0,24,12,13,49,1,24,25,36,37,49,281.632653,481.632653,1081.632653,24.0,0.0,0.0,1.0,0.0,0.0,24.0,24.0,0_10_pct,6759.183673,2400.0,9159.183673,0.097959
21,677764,Italian American Civil Rights League @MS 366,SACC,11236,40.647388,-73.891883,0,180,0,18,9,9,36,1,18,19,27,28,36,311.111111,511.111111,1111.111111,18.0,0.0,0.0,1.0,0.0,0.0,18.0,18.0,0_10_pct,5600.000000,1800.0,7400.000000,0.100000
30,791237,Yeshiva Kehilath Yakov,SACC,11211,40.707164,-73.959423,0,284,0,28,14,14,56,1,28,29,42,43,56,270.422535,470.422535,1070.422535,28.0,0.0,0.0,1.0,0.0,0.0,28.0,28.0,0_10_pct,7571.830986,2800.0,10371.830986,0.098592
31,818672,"Grand Street Settlement, Inc.",SACC,10038,40.712360,-73.998909,0,100,0,10,5,5,20,1,10,11,15,16,20,400.000000,600.000000,1200.000000,10.0,0.0,0.0,1.0,0.0,0.0,10.0,10.0,0_10_pct,4000.000000,1000.0,5000.000000,0.100000


In [21]:
if has_solution:
    build_rows = []
    for (l, s) in cand_key_tuples:
        if y[l, s].X > 0.5:
            row = cand_df[(cand_df["candidate_id"] == l) & (cand_df["size"] == s)].iloc[0].to_dict()
            row["build_selected"] = int(round(y[l, s].X))
            row["new_under5_assigned"] = g[l, s].X
            row["under5_equipment_cost_newbuild"] = 100 * g[l, s].X
            row["total_newbuild_related_cost"] = row["fixed_build_cost"] + row["under5_equipment_cost_newbuild"]
            build_rows.append(row)

    new_build_solution = pd.DataFrame(build_rows)

    print("Selected new builds:", new_build_solution.shape)
    display(new_build_solution.head())

Selected new builds: (1323, 14)


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost,blocked_by_existing,build_selected,new_under5_assigned,under5_equipment_cost_newbuild,total_newbuild_related_cost
0,52,10001,40.740011,-74.004994,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
1,93,10001,40.740448,-74.003572,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
2,126,10002,40.722981,-73.982954,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
3,129,10002,40.708740,-73.988030,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
4,136,10002,40.723179,-73.993990,False,large,400,200,115000,0,1,200.0,20000.0,135000.0


In [22]:
if has_solution:
    if not facility_solution.empty:
        facility_expand_zip = (
            facility_solution.groupby("zipcode", as_index=False)
            .agg(
                expanded_total=("expand_total", "sum"),
                expanded_under5=("u_under5", "sum"),
                expansion_cost=("expansion_cost", "sum"),
                under5_equipment_cost_expansion=("under5_equipment_cost_expansion", "sum"),
                total_expansion_related_cost=("total_expansion_related_cost", "sum")
            )
        )
    else:
        facility_expand_zip = pd.DataFrame(columns=[
            "zipcode", "expanded_total", "expanded_under5", "expansion_cost",
            "under5_equipment_cost_expansion", "total_expansion_related_cost"
        ])

    if not new_build_solution.empty:
        new_build_zip = (
            new_build_solution.groupby("zipcode", as_index=False)
            .agg(
                n_new_facilities=("candidate_id", "count"),
                new_total_capacity=("new_total_capacity", "sum"),
                new_under5_capacity=("new_under5_assigned", "sum"),
                new_build_cost=("fixed_build_cost", "sum"),
                under5_equipment_cost_newbuild=("under5_equipment_cost_newbuild", "sum"),
                total_newbuild_related_cost=("total_newbuild_related_cost", "sum")
            )
        )
    else:
        new_build_zip = pd.DataFrame(columns=[
            "zipcode", "n_new_facilities", "new_total_capacity", "new_under5_capacity",
            "new_build_cost", "under5_equipment_cost_newbuild", "total_newbuild_related_cost"
        ])

    zipcode_solution = zip_df.copy()
    zipcode_solution = zipcode_solution.merge(facility_expand_zip, on="zipcode", how="left")
    zipcode_solution = zipcode_solution.merge(new_build_zip, on="zipcode", how="left")

    fill_zero_cols = [
        "expanded_total", "expanded_under5", "expansion_cost", "under5_equipment_cost_expansion",
        "total_expansion_related_cost", "n_new_facilities", "new_total_capacity", "new_under5_capacity",
        "new_build_cost", "under5_equipment_cost_newbuild", "total_newbuild_related_cost"
    ]
    for c in fill_zero_cols:
        zipcode_solution[c] = zipcode_solution[c].fillna(0)

    zipcode_solution["final_total_capacity"] = (
        zipcode_solution["existing_total"]
        + zipcode_solution["expanded_total"]
        + zipcode_solution["new_total_capacity"]
    )

    zipcode_solution["final_under5_capacity"] = (
        zipcode_solution["existing_under5"]
        + zipcode_solution["expanded_under5"]
        + zipcode_solution["new_under5_capacity"]
    )

    zipcode_solution["total_slack"] = zipcode_solution["final_total_capacity"] - zipcode_solution["req_total"]
    zipcode_solution["under5_slack"] = zipcode_solution["final_under5_capacity"] - zipcode_solution["req_under5"]

    zipcode_solution["all_requirements_met_after"] = (
        (zipcode_solution["total_slack"] >= -1e-6) &
        (zipcode_solution["under5_slack"] >= -1e-6)
    ).astype(int)

    display(zipcode_solution.head())

,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before,req_total_original,req_under5_original,zero_child_population_adjustment,expanded_total,expanded_under5,expansion_cost,under5_equipment_cost_expansion,total_expansion_related_cost,n_new_facilities,new_total_capacity,new_under5_capacity,new_build_cost,under5_equipment_cost_newbuild,total_newbuild_related_cost,final_total_capacity,final_under5_capacity,total_slack,under5_slack,all_requirements_met_after
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609,24,9.0,58,472,1,1,1,667,496,0,72.0,72.0,36353.135326,7200.0,43553.135326,2.0,800.0,400.0,230000.0,40000.0,270000.0,1481.0,496.0,814.0,0.0,1
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724,198,56.0,0,1230,0,1,1,3394,1428,0,430.0,430.0,142424.102639,43000.0,185424.102639,4.0,1600.0,800.0,460000.0,80000.0,540000.0,6754.0,1428.0,3360.0,0.0,1
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995,0,7.0,0,960,0,1,1,984,960,0,160.0,160.0,39762.682751,16000.0,55762.682751,4.0,1600.0,800.0,460000.0,80000.0,540000.0,3755.0,960.0,2771.0,0.0,1
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263,0,2.0,0,289,0,1,1,232,289,0,39.0,39.0,22836.923077,3900.0,26736.923077,2.0,500.0,250.0,180000.0,25000.0,205000.0,802.0,289.0,570.0,0.0,1
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39,0,1.0,363,323,1,1,1,402,323,0,0.0,0.0,0.000000,0.0,0.000000,2.0,800.0,323.0,230000.0,32300.0,262300.0,839.0,323.0,437.0,0.0,1


In [23]:
if has_solution:
    objective_breakdown = pd.DataFrame({
        "component": [
            "expansion_cost",
            "new_build_cost",
            "under5_equipment_cost",
            "total_objective"
        ],
        "value": [
            expansion_cost_expr.getValue(),
            new_build_cost_expr.getValue(),
            under5_equipment_cost_expr.getValue(),
            model.ObjVal
        ]
    })

    display(objective_breakdown)

,component,value
0,expansion_cost,6.168863e+06
1,new_build_cost,1.513350e+08
2,under5_equipment_cost,2.777210e+07
3,total_objective,1.852760e+08


In [24]:
if has_solution:
    run_metadata = pd.DataFrame({
        "metric": [
            "model_status",
            "objective_value",
            "mip_gap",
            "solve_seconds",
            "num_variables",
            "num_constraints",
            "solution_count"
        ],
        "value": [
            model.Status,
            model.ObjVal,
            model.MIPGap if hasattr(model, "MIPGap") else np.nan,
            solve_seconds,
            model.NumVars,
            model.NumConstrs,
            model.SolCount
        ]
    })

    summary_stats = pd.DataFrame({
        "metric": [
            "n_expanded_facilities",
            "total_expansion_slots",
            "total_expansion_under5_slots",
            "n_new_facilities",
            "total_new_capacity",
            "total_new_under5_capacity",
            "zipcodes_meeting_all_requirements_after"
        ],
        "value": [
            len(facility_solution),
            facility_solution["expand_total"].sum() if not facility_solution.empty else 0,
            facility_solution["u_under5"].sum() if not facility_solution.empty else 0,
            len(new_build_solution) if not new_build_solution.empty else 0,
            new_build_solution["new_total_capacity"].sum() if not new_build_solution.empty else 0,
            new_build_solution["new_under5_assigned"].sum() if not new_build_solution.empty else 0,
            zipcode_solution["all_requirements_met_after"].sum()
        ]
    })

    display(run_metadata)
    display(summary_stats)

,metric,value
0,model_status,9.000000e+00
1,objective_value,1.852760e+08
2,mip_gap,1.422223e-03
3,solve_seconds,6.006181e+02
4,num_variables,2.251160e+05
5,num_constraints,1.866670e+05
6,solution_count,1.000000e+01


,metric,value
0,n_expanded_facilities,982.0
1,total_expansion_slots,17450.0
2,total_expansion_under5_slots,17410.0
3,n_new_facilities,1323.0
4,total_new_capacity,523300.0
5,total_new_under5_capacity,260311.0
6,zipcodes_meeting_all_requirements_after,311.0


In [25]:
if has_solution:
    facility_solution.to_csv(SOLUTIONS_DIR / "facility_solution_realistic.csv", index=False)
    zipcode_solution.to_csv(SOLUTIONS_DIR / "zipcode_solution_realistic.csv", index=False)
    objective_breakdown.to_csv(SOLUTIONS_DIR / "objective_breakdown_realistic.csv", index=False)
    run_metadata.to_csv(SOLUTIONS_DIR / "run_metadata_realistic.csv", index=False)
    summary_stats.to_csv(SOLUTIONS_DIR / "summary_stats_realistic.csv", index=False)

    if not new_build_solution.empty:
        new_build_solution.to_csv(SOLUTIONS_DIR / "new_build_solution_realistic.csv", index=False)
    else:
        pd.DataFrame(columns=cand_df.columns.tolist() + [
            "build_selected",
            "new_under5_assigned",
            "under5_equipment_cost_newbuild",
            "total_newbuild_related_cost"
        ]).to_csv(SOLUTIONS_DIR / "new_build_solution_realistic.csv", index=False)

    print("Realistic solution files saved to:", SOLUTIONS_DIR)

Realistic solution files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/solutions
